In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("netflix_clean.csv")
print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (8803, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
#checking missing values in each columns
print("Missing Values Per Column:")
print(df.isnull().sum())

Missing Values Per Column:
show_id            0
type               0
title              0
director        2633
cast             825
country          830
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64


In [4]:
# Handling missing values
# replace missing values with 'Unknown' in the selected columns.
columns_to_fill = ['director', 'cast', 'country', 'rating']
for column in columns_to_fill:
    df[column] = df[column].fillna('Unknown')

# display missing values after handling
print("Missing Values After Handling:")
print(df.isnull().sum())

Missing Values After Handling:
show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         0
listed_in        0
description      0
dtype: int64


In [5]:
#final duplicate check
#check for exact duplicate rows
print("Exact Duplicate Rows:", df.duplicated().sum())

#Check for duplicate titles with the same type and release year
print(
    "Duplicate Title-Type-Year Records:",
    df.duplicated(subset=['title', 'type', 'release_year']).sum()
)

Exact Duplicate Rows: 0
Duplicate Title-Type-Year Records: 0


In [6]:
#final dataset validation
expected_columns = [
    'show_id', 'type', 'title', 'director', 'cast', 'country',
    'date_added', 'release_year', 'rating', 'duration',
    'listed_in', 'description'
]

print("Final Dataset Shape:", df.shape)
print("All Expected Columns Present:", list(df.columns) == expected_columns)
print("Total Missing Values:", df.isnull().sum().sum())

Final Dataset Shape: (8803, 12)
All Expected Columns Present: True
Total Missing Values: 10


In [7]:
# save the final cleaned dataset
df.to_csv("netflix_final_cleaned.csv", index=False)
print("Final cleaned dataset saved successfully!")

Final cleaned dataset saved successfully!


In [9]:
#upload the final cleaned dataset to GitHub

import requests
import base64
from google.colab import userdata

#gitHub repository details
OWNER = "pradnyayuvrajkolekar2"
REPO = "recommendation-system"
BRANCH = "main"

#destination path in GitHub
FILE_PATH = "output/netflix_final_cleaned.csv"

#get gitHub token securely from Colab Secrets
TOKEN = userdata.get("GH_TOKEN")

if not TOKEN:
    raise ValueError("GitHub token not found. Check Colab Secrets.")

#read the final cleaned CSV file
with open("netflix_final_cleaned.csv", "rb") as file:
    content = base64.b64encode(file.read()).decode("utf-8")

#gitHub API URL
url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{FILE_PATH}"

headers = {
    "Authorization": f"token {TOKEN}",
    "Accept": "application/vnd.github+json"
}

#check whether the file already exists
response = requests.get(
    url,
    headers=headers,
    params={"ref": BRANCH}
)

#prepare upload details
data = {
    "message": "Updated final cleaned Netflix dataset",
    "content": content,
    "branch": BRANCH
}

if response.status_code == 200:
    #include SHA when updating an existing file
    data["sha"] = response.json()["sha"]
elif response.status_code != 404:
    raise Exception(f"GitHub error: {response.text}")

#upload or update the CSV file
upload = requests.put(url, headers=headers, json=data)

if upload.status_code in [200, 201]:
    print("Final dataset successfully uploaded to GitHub!")
    print("GitHub file:", upload.json()["content"]["path"])
else:
    raise Exception(f"Upload failed: {upload.text}")

Final dataset successfully uploaded to GitHub!
GitHub file: output/netflix_final_cleaned.csv
